# 004 — Features (MovieLens, Feast offline join)

Port of the reference `src/feature_engineer/001-features.ipynb`, adapted to
MovieLens and the local Feast repo (`feature/feature_store`).

Reads the interaction parquets from `feature/output/` (produced by `003`),
pulls **movie** and **user** aggregate features from Feast via point-in-time
`get_historical_features`, maps raw ids to dense indices (`IDMapper`), builds
the recent-10 `item_sequence`, fits an sklearn `ColumnTransformer`
(title TF-IDF + genres count-vectorizer + rating-aggregate StandardScaler),
and persists `train_features.parquet` / `val_features.parquet` +
`idm.json` + `item_metadata_pipeline.dill` to `feature/output/engineer/`.

In [1]:
import os
import json

import numpy as np
import pandas as pd
import dill
from dotenv import load_dotenv
from loguru import logger
from pydantic import BaseModel
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import StandardScaler

_PROJECT_ROOT = os.path.abspath(os.path.join(os.getcwd(), "..", ".."))
load_dotenv(os.path.join(_PROJECT_ROOT, ".env"))
import sys
sys.path.insert(0, _PROJECT_ROOT)

from feature.id_mapper import IDMapper, map_indice
from feature.features.tfm import title_pipeline_steps, genres_pipeline_steps, rating_agg_pipeline_steps


class Args(BaseModel):
    run_name: str = "004-features"
    random_seed: int = int(os.getenv("RANDOM_SEED", "42"))
    user_col: str = "userId"
    item_col: str = "movieId"
    rating_col: str = "rating"
    timestamp_col: str = "event_timestamp"
    sequence_length: int = 10
    windows_days: list = [90, 30, 7]
    # populated by init()
    project_root: str = ""
    output_dir: str = ""
    engineer_dir: str = ""

    def init(self):
        self.project_root = _PROJECT_ROOT
        self.output_dir = os.path.join(self.project_root, "feature", "output")
        self.engineer_dir = os.path.join(self.output_dir, "engineer")
        os.makedirs(self.engineer_dir, exist_ok=True)
        return self


args = Args().init()
logger.info(f"output_dir={args.output_dir}  engineer_dir={args.engineer_dir}")
print(json.dumps({"sequence_length": args.sequence_length, "windows_days": args.windows_days}, indent=2))

2026-07-16 00:43:07.833 | INFO     | __main__:<module>:46 - output_dir=E:\MachineLearning\Recommendation_System\feature\output  engineer_dir=E:\MachineLearning\Recommendation_System\feature\output\engineer


{
  "sequence_length": 10,
  "windows_days": [
    90,
    30,
    7
  ]
}


## Load interactions + IDMapper

Concatenate train/val, fit the id mapper on sorted train ids, attach dense
`user_indice` / `item_indice`.

In [2]:
train_df = pd.read_parquet(os.path.join(args.output_dir, "train.parquet"))
val_df = pd.read_parquet(os.path.join(args.output_dir, "val.parquet"))
logger.info(f"train={train_df.shape}  val={val_df.shape}")

full_df = pd.concat([train_df, val_df], ignore_index=True)
unique_user_ids = sorted(train_df[args.user_col].unique())
unique_item_ids = sorted(train_df[args.item_col].unique())
idm = IDMapper().fit(unique_user_ids, unique_item_ids)
logger.info(f"IDMapper: users={len(unique_user_ids)}  items={len(unique_item_ids)}")

full_df = map_indice(full_df, idm, args.user_col, args.item_col)
logger.info(f"full_df={full_df.shape}  cols={list(full_df.columns)}")
full_df.head()

2026-07-16 00:43:07.918 | INFO     | __main__:<module>:3 - train=(79425, 5)  val=(103, 5)


2026-07-16 00:43:07.925 | INFO     | __main__:<module>:9 - IDMapper: users=578  items=2264


2026-07-16 00:43:08.017 | INFO     | __main__:<module>:12 - full_df=(79528, 7)  cols=['userId', 'movieId', 'rating', 'event_timestamp', 'source', 'user_indice', 'item_indice']


,userId,movieId,rating,event_timestamp,source,user_indice,item_indice
0,429,595,5.0,1996-03-29 18:36:55,train,408,280
1,429,225,4.0,1996-03-29 18:36:55,train,408,111
2,429,222,4.0,1996-03-29 18:36:55,train,408,108
3,429,218,4.0,1996-03-29 18:36:55,train,408,107
4,429,349,3.0,1996-03-29 18:36:55,train,408,173


## Merge movie features

`003` already computed per-`(movieId, event_timestamp)` aggregates, so a plain
left-merge on `(movieId, event_timestamp)` is an exact point-in-time match — no
asof needed. (Feast's file offline store wraps this in a dask asof-join that
OOMs at 79k rows; the Feast repo stays for online/Redis serving only.)

In [3]:
movie_stats = pd.read_parquet(os.path.join(args.output_dir, "movie_rating_stats.parquet"))
logger.info(f"movie_stats={movie_stats.shape}  cols={list(movie_stats.columns)}")

full_features_df = full_df.merge(movie_stats, how="left", on=[args.item_col, args.timestamp_col], validate="m:1")
logger.info(f"after movie merge: {full_features_df.shape}")
assert full_features_df["title"].notna().all(), "movie merge left nulls"
full_features_df.head()

2026-07-16 00:43:08.059 | INFO     | __main__:<module>:2 - movie_stats=(79528, 10)  cols=['movieId', 'event_timestamp', 'title', 'genres', 'movie_rating_cnt_90d', 'movie_rating_avg_prev_rating_90d', 'movie_rating_cnt_30d', 'movie_rating_avg_prev_rating_30d', 'movie_rating_cnt_7d', 'movie_rating_avg_prev_rating_7d']


2026-07-16 00:43:08.125 | INFO     | __main__:<module>:5 - after movie merge: (79528, 15)


,userId,movieId,rating,event_timestamp,source,user_indice,item_indice,title,genres,movie_rating_cnt_90d,movie_rating_avg_prev_rating_90d,movie_rating_cnt_30d,movie_rating_avg_prev_rating_30d,movie_rating_cnt_7d,movie_rating_avg_prev_rating_7d
0,429,595,5.0,1996-03-29 18:36:55,train,408,280,Beauty and the Beast (1991),Animation|Children|Fantasy|Musical|Romance|IMAX,0,0.0,0,0.0,0,0.0
1,429,225,4.0,1996-03-29 18:36:55,train,408,111,Disclosure (1994),Drama|Thriller,0,0.0,0,0.0,0,0.0
2,429,222,4.0,1996-03-29 18:36:55,train,408,108,Circle of Friends (1995),Drama|Romance,0,0.0,0,0.0,0,0.0
3,429,218,4.0,1996-03-29 18:36:55,train,408,107,Boys on the Side (1995),Comedy|Drama,0,0.0,0,0.0,0,0.0
4,429,349,3.0,1996-03-29 18:36:55,train,408,173,Clear and Present Danger (1994),Action|Crime|Drama|Thriller,0,0.0,0,0.0,0,0.0


## Merge user features + build item_sequence

User 90-day aggregates, the recent-10 movie list, and the padded timestamp
sequence + buckets — same exact `(userId, event_timestamp)` match. `item_sequence`
converts the recent-10 movie ids to dense indices (left-padded -1).

In [4]:
user_stats = pd.read_parquet(os.path.join(args.output_dir, "user_rating_stats.parquet"))
# MovieLens records batch ratings with the same (userId, event_timestamp) second.
# 003 emits one row per event; collapse to one per (userId, event_timestamp)
# keeping the LAST event (the most complete recent-10 list) so the join is m:1.
user_stats = user_stats.drop_duplicates([args.user_col, args.timestamp_col], keep="last").reset_index(drop=True)
logger.info(f"user_stats (deduped)={user_stats.shape}  cols={list(user_stats.columns)}")

full_features_df = full_features_df.merge(user_stats, how="left", on=[args.user_col, args.timestamp_col], validate="m:1")
logger.info(f"after user merge: {full_features_df.shape}  cols={list(full_features_df.columns)}")
assert full_features_df["user_rating_cnt_90d"].notna().all(), "user merge left nulls"


def convert_movie_to_idx(inp, sequence_length=10, padding_value=-1):
    if inp is None or (isinstance(inp, float) and np.isnan(inp)) or str(inp).strip() == "":
        return [padding_value] * sequence_length
    movie_ids = [int(x) for x in str(inp).split(",") if x.strip()]
    indices = [idm.get_item_index(mid) for mid in movie_ids]
    pad_needed = sequence_length - len(indices)
    if pad_needed > 0:
        indices = [padding_value] * pad_needed + indices
    return indices[:sequence_length]


full_features_df = full_features_df.assign(
    item_sequence=lambda d: d["user_rating_list_10_recent_movie"].apply(convert_movie_to_idx)
)
logger.info("built item_sequence")
full_features_df[["user_rating_list_10_recent_movie", "item_sequence"]].head()

2026-07-16 00:43:08.321 | INFO     | __main__:<module>:6 - user_stats (deduped)=(66896, 8)  cols=['userId', 'event_timestamp', 'user_rating_cnt_90d', 'user_rating_avg_prev_rating_90d', 'user_rating_list_10_recent_movie', 'user_rating_list_10_recent_movie_timestamp', 'item_sequence_ts', 'item_sequence_ts_bucket']


2026-07-16 00:43:08.406 | INFO     | __main__:<module>:9 - after user merge: (79528, 21)  cols=['userId', 'movieId', 'rating', 'event_timestamp', 'source', 'user_indice', 'item_indice', 'title', 'genres', 'movie_rating_cnt_90d', 'movie_rating_avg_prev_rating_90d', 'movie_rating_cnt_30d', 'movie_rating_avg_prev_rating_30d', 'movie_rating_cnt_7d', 'movie_rating_avg_prev_rating_7d', 'user_rating_cnt_90d', 'user_rating_avg_prev_rating_90d', 'user_rating_list_10_recent_movie', 'user_rating_list_10_recent_movie_timestamp', 'item_sequence_ts', 'item_sequence_ts_bucket']


2026-07-16 00:43:09.082 | INFO     | __main__:<module>:27 - built item_sequence


,user_rating_list_10_recent_movie,item_sequence
0,"420,590,349,227,225,222,218,165,161,150","[206, 276, 173, 112, 111, 108, 107, 82, 78, 70]"
1,"420,590,349,227,225,222,218,165,161,150","[206, 276, 173, 112, 111, 108, 107, 82, 78, 70]"
2,"420,590,349,227,225,222,218,165,161,150","[206, 276, 173, 112, 111, 108, 107, 82, 78, 70]"
3,"420,590,349,227,225,222,218,165,161,150","[206, 276, 173, 112, 111, 108, 107, 82, 78, 70]"
4,"420,590,349,227,225,222,218,165,161,150","[206, 276, 173, 112, 111, 108, 107, 82, 78, 70]"


## Split back to train / val + persist

Same temporal split as `003`: `event_timestamp < val_timestamp` -> train.

In [5]:
val_timestamp = val_df[args.timestamp_col].min()
logger.info(f"val_timestamp={val_timestamp}")

train_out = full_features_df.loc[lambda d: d[args.timestamp_col] < val_timestamp].copy()
val_out = full_features_df.loc[lambda d: d[args.timestamp_col] >= val_timestamp].copy()
logger.info(f"train_out={train_out.shape}  val_out={val_out.shape}")
assert train_out.shape[0] == len(train_df), "train row count drifted"
assert val_out.shape[0] == len(val_df), "val row count drifted"


def check_dup(df):
    n = df[[args.user_col, args.item_col, args.timestamp_col]].duplicated().sum()
    assert n == 0, f"{n} duplicates"


check_dup(train_out)
check_dup(val_out)

train_fp = os.path.join(args.engineer_dir, "train_features.parquet")
val_fp = os.path.join(args.engineer_dir, "val_features.parquet")
idm_fp = os.path.join(args.engineer_dir, "idm.json")
train_out.to_parquet(train_fp, index=False)
val_out.to_parquet(val_fp, index=False)
idm.save(idm_fp)
logger.info(f"wrote {train_fp}  {val_fp}  {idm_fp}")

2026-07-16 00:43:09.109 | INFO     | __main__:<module>:2 - val_timestamp=2018-06-26 21:04:28


2026-07-16 00:43:09.150 | INFO     | __main__:<module>:6 - train_out=(79425, 22)  val_out=(103, 22)


2026-07-16 00:43:09.565 | INFO     | __main__:<module>:25 - wrote E:\MachineLearning\Recommendation_System\feature\output\engineer\train_features.parquet  E:\MachineLearning\Recommendation_System\feature\output\engineer\val_features.parquet  E:\MachineLearning\Recommendation_System\feature\output\engineer\idm.json


## Fit sklearn metadata pipeline

`ColumnTransformer` over title (TF-IDF), genres (multi-label count vectorizer),
and the 6 movie rating aggregates (StandardScaler). Fit on unique movies in
train, persist with `dill` for reuse at training/serving time.

In [6]:
rating_agg_cols = [c for d in args.windows_days for c in (f"movie_rating_cnt_{d}d", f"movie_rating_avg_prev_rating_{d}d")]
logger.info(f"rating_agg_cols={rating_agg_cols}")

tfm = [
    ("title", Pipeline(title_pipeline_steps()), ["title"]),
    ("genres", Pipeline(genres_pipeline_steps()), "genres"),
    ("rating_agg", Pipeline(rating_agg_pipeline_steps()), rating_agg_cols),
]

preprocessing = ColumnTransformer(transformers=tfm, remainder="drop")
item_metadata_pipeline = Pipeline(steps=[("preprocessing", preprocessing), ("normalizer", StandardScaler())])

fit_df = train_out.drop_duplicates(subset=[args.item_col])
item_metadata_pipeline.fit(fit_df)
logger.info(f"fit pipeline on {len(fit_df)} unique movies")

dill_fp = os.path.join(args.engineer_dir, "item_metadata_pipeline.dill")
with open(dill_fp, "wb") as f:
    dill.dump(item_metadata_pipeline, f)
logger.info(f"wrote {dill_fp}")

2026-07-16 00:43:09.578 | INFO     | __main__:<module>:2 - rating_agg_cols=['movie_rating_cnt_90d', 'movie_rating_avg_prev_rating_90d', 'movie_rating_cnt_30d', 'movie_rating_avg_prev_rating_30d', 'movie_rating_cnt_7d', 'movie_rating_avg_prev_rating_7d']


2026-07-16 00:43:10.029 | INFO     | __main__:<module>:15 - fit pipeline on 2264 unique movies


2026-07-16 00:43:10.046 | INFO     | __main__:<module>:20 - wrote E:\MachineLearning\Recommendation_System\feature\output\engineer\item_metadata_pipeline.dill


## Verify

Sanity-check shapes, nulls on key columns, and that val items/users are all in train (no cold-start).

In [7]:
key_cols = [args.user_col, args.item_col, "user_indice", "item_indice", "item_sequence", "item_sequence_ts_bucket", args.rating_col, args.timestamp_col]
nulls = train_out[key_cols].isna().sum().sum() + val_out[key_cols].isna().sum().sum()
logger.info(f"key-col nulls={nulls}")
assert nulls == 0, "nulls in key columns"

train_items = set(train_out[args.item_col].unique())
val_items = set(val_out[args.item_col].unique())
missing = val_items - train_items
logger.info(f"val items not in train: {len(missing)}")
assert not missing, f"{len(missing)} cold-start val items"

logger.info(f"train_features={train_out.shape}  val_features={val_out.shape}  cols={list(train_out.columns)}")
logger.info("004 verification OK")

2026-07-16 00:43:10.070 | INFO     | __main__:<module>:3 - key-col nulls=0


2026-07-16 00:43:10.074 | INFO     | __main__:<module>:9 - val items not in train: 0


2026-07-16 00:43:10.076 | INFO     | __main__:<module>:12 - train_features=(79425, 22)  val_features=(103, 22)  cols=['userId', 'movieId', 'rating', 'event_timestamp', 'source', 'user_indice', 'item_indice', 'title', 'genres', 'movie_rating_cnt_90d', 'movie_rating_avg_prev_rating_90d', 'movie_rating_cnt_30d', 'movie_rating_avg_prev_rating_30d', 'movie_rating_cnt_7d', 'movie_rating_avg_prev_rating_7d', 'user_rating_cnt_90d', 'user_rating_avg_prev_rating_90d', 'user_rating_list_10_recent_movie', 'user_rating_list_10_recent_movie_timestamp', 'item_sequence_ts', 'item_sequence_ts_bucket', 'item_sequence']


2026-07-16 00:43:10.076 | INFO     | __main__:<module>:13 - 004 verification OK
